# Object Detection

Detect objects in the camera image with **YOLOv8** and chase one: steer toward the biggest detection of a target class (default: `sports ball`), stop when close, spin to search when it is lost. The pretrained model ships in `assets/object-detection/models/` — YOLOv8n trained on COCO (80 classes), exported to NCNN, the fastest backend on the robot's CPU.

## [1] Setup

### 1.1. Talking to the car

Two helpers are the whole robot interface — the camera in, the wheels out. The API takes radians; we work in degrees (+ = left).

In [ ]:
import os

DIR = "assets/object-detection"
if not os.path.isdir(DIR):               # kernel cwd is the workspace root
    DIR = "examples/" + DIR

import math

import cv2
import numpy as np
import requests

BASE_URL = "http://localhost"


def camera():
    """Latest camera frame as a BGR image (ndarray)."""
    jpg = requests.get(f"{BASE_URL}/camera", timeout=2).content
    return cv2.imdecode(np.frombuffer(jpg, np.uint8), cv2.IMREAD_COLOR)


def drive(speed, steering):
    """speed in m/s, steering in degrees (+ = left). The API wants radians."""
    requests.post(f"{BASE_URL}/speed", json={"value": float(speed)}, timeout=2)
    requests.post(f"{BASE_URL}/steering",
                  json={"value": math.radians(steering)}, timeout=2)

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(cv2.cvtColor(camera(), cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

## [2] Detection

### 2.1. The detector

Load the model and see the 80 COCO classes — pick any of them as `TARGET` (`person`, `dog`, `cup`, ...).

In [ ]:
from ultralytics import YOLO

TARGET = "sports ball"     # any COCO class name

yolo = YOLO(f"{DIR}/models/yolov8n_ncnn_model", task="detect")
ids = {name: i for i, name in yolo.names.items()}
print(sorted(ids))

### 2.2. Detect in one frame

We look for the **target class only** (`classes=[...]`): picking each box's top class instead would lose the target up close, where a lookalike class starts to outscore it. `conf=0.15` accepts weak detections for the same reason. `imgsz=320` — a ball doesn't need 640, and it's ~4x less compute.

In [ ]:
IMG_SIZE = 320
CONF = 0.15

img = camera()
boxes = yolo(img, imgsz=IMG_SIZE, conf=CONF,
             classes=[ids[TARGET]], verbose=False)[0].boxes
for box in boxes:
    x1, y1, x2, y2 = (int(v) for v in box.xyxy[0].tolist())
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 255), 2)
    cv2.putText(img, f"{TARGET} {float(box.conf[0]):.2f}", (x1, y1 - 6),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
print(len(boxes), "detection(s)")
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

## [3] Chase

### 3.1. The control policy

From the biggest box we take its center → normalized offset from image center (−1..1) → proportional steering, clamped to the ±20° hardware limit. Speed is all-or-nothing: chase at `SPEED` until the box fills `NEAR_HEIGHT` of the frame, then stop — arrived.

In [ ]:
SPEED = 0.8            # m/s while chasing
STEER_GAIN = 30.0
NEAR_HEIGHT = 0.45     # stop when the target fills this fraction of the frame
LOST_TIMEOUT = 1.0     # keep the last sighting this long before searching

### 3.2. The live view

The web view is plumbing, not the logic — run the cell and forget it (the pages themselves live in the example's folder). It skips itself with a notice when port 5000 is already taken by another app.

In [ ]:
"""Live web view (MYAPP tab, port 5000): the camera frame with every YOLO
detection drawn on it -- the target class highlighted."""
import socket
import threading

_server = [None]     # the one live web view — a new serve() replaces it


def stop_view():
    if _server[0]:
        _server[0].shutdown()
        _server[0] = None


def _start(app):
    import logging
    logging.getLogger("werkzeug").setLevel(logging.ERROR)
    from werkzeug.serving import make_server
    stop_view()                      # one view at a time — replace the old one
    try:
        probe = socket.socket()                       # quiet pre-check: werkzeug
        probe.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)   # prints and
        probe.bind(("", 5000))                        # exits on a busy port
        probe.close()
        server = make_server("0.0.0.0", 5000, app, threaded=True)
    except (OSError, SystemExit):
        print("web view: port 5000 is in use by another example — "
              "interrupt (⏹) that cell or restart its kernel; "
              "continuing without the live view")
        return None
    _server[0] = server
    threading.Thread(target=server.serve_forever, daemon=True).start()
    print("web view: open the MYAPP tab to watch")
    return server

_frame = [b""]          # latest annotated camera frame as JPEG
_data = [{}]            # latest status


def update(img_bgr, detections, target, status):
    """detections: list of (class name, confidence, (x1, y1, x2, y2))."""
    img = img_bgr.copy()
    for name, conf, (x1, y1, x2, y2) in detections:
        hit = name == target
        color = (90, 200, 255) if hit else (90, 90, 90)
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), color,
                      2 if hit else 1)
        cv2.putText(img, f"{name} {conf:.2f}", (int(x1), int(y1) - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
    _frame[0] = cv2.imencode(".jpg", img)[1].tobytes()
    _data[0] = {"status": status}


def serve():
    from flask import Flask, Response, jsonify
    app = Flask(__name__)
    page = open(f"{DIR}/webui.html").read()   # presentation lives in the folder

    @app.get("/")
    def index():
        return page

    @app.get("/frame")
    def frame():
        return Response(_frame[0], mimetype="image/jpeg",
                        headers={"Cache-Control": "no-store"})

    @app.get("/data")
    def data():
        return jsonify(_data[0])

    return _start(app)

### 3.3. Chase!

Runs until you interrupt the cell (⏹) — the `finally` block stops the car.

In [ ]:
import time

serve()   # MYAPP tab: camera + detections, live
last_best, last_seen = None, 0.0
try:
    while True:
        t0 = time.time()
        img = camera()
        h, w = img.shape[:2]

        best, dets = None, []
        for box in yolo(img, imgsz=IMG_SIZE, conf=CONF,
                        classes=[ids[TARGET]], verbose=False)[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            dets.append((TARGET, float(box.conf[0]), (x1, y1, x2, y2)))
            if best is None or (x2 - x1) * (y2 - y1) > best[0]:
                best = ((x2 - x1) * (y2 - y1), (x1 + x2) / 2, y2 - y1)

        if best is not None:
            last_best, last_seen = best, time.time()
        elif time.time() - last_seen < LOST_TIMEOUT:
            best = last_best              # one flickered frame != lost

        if best is None:
            drive(0.0, 15)                              # look around
            status = "searching..."
        else:
            _, cx, box_h = best
            offset = (cx - w / 2) / (w / 2)             # -1 .. 1
            steering = max(-20, min(20, -offset * STEER_GAIN))
            speed = 0.0 if box_h / h > NEAR_HEIGHT else SPEED
            drive(speed, steering)
            status = (f"tracking  offset {offset:+.2f}  "
                      + ("arrived" if speed == 0 else f"speed {speed}"))
        update(img, dets, TARGET, status)

        time.sleep(max(0.0, 1 / 10 - (time.time() - t0)))
except KeyboardInterrupt:
    pass
finally:
    drive(0, 0)
    stop_view()   # the live view lives and dies with this cell
    print("stopped")